### Schmidt-Weighted Truncation

#### Rigorous Replication Protocol

- **N_TRIALS = 100** per configuration (bootstrap-stable statistics)
- Circuits: **UCCSD, Haar L=3/5, Brickwork L=3/5** (adds missing family)
- Statistics: **geometric mean + 95% bootstrap CI** for ratios; **median + IQR** for fidelity
- **Paired per-trial comparison**: same exact circuit instance and exact statevector for all 4 methods
- **Layer-by-layer tracking**: γ² per gate to validate the compounding narrative
- **Precise implementation** of Schmidt-1cut and Schmidt-3cut (fixed, documented)


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats as sci_stats


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

THIS_DIR = REPO_ROOT / "tests"
FIGURE_DIR = THIS_DIR / "figures"
DATA_DIR = THIS_DIR / "results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

from src.experimentation.runner import (
    exact_statevector,
    make_brickwork,
    make_haar,
    make_uccsd,
    trial_stats,
    ratio_stats,
    bootstrap_ci,
)
from src.core.sparse_state import (
    apply_1qubit_gate_kernel_opt,
    apply_2qubit_gate_kernel_opt,
    merge_sorted_kernel,
    radix_sort_uint64,
)
from src.benchmarking.fidelity import compute_fidelity
from src.visualization.style import mplstyle, polish_axes, save_figure

mplstyle(dpi=180)
mpl.rcParams.update(
    {
        "font.size": 8.0,
        "axes.labelsize": 8.0,
        "xtick.labelsize": 7.0,
        "ytick.labelsize": 7.0,
        "legend.fontsize": 6.8,
        "lines.linewidth": 1.2,
        "lines.markersize": 3.0,
        "axes.linewidth": 0.8,
    }
)
print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ── Benchmark parameters ──────────────────────────────────────────────────────
N = 16  # system size (matches original paper)
K = 8192  # sparse budget (matches original paper)
N_TRIALS = 100  # trials per configuration (original used 3 — statistically meaningless)
SEED_BASE = 20260520
FORCE_RERUN = False

# Circuit families
CIRCUIT_CONFIGS = [
    {
        "key": "haar_L3",
        "label": "Haar L=3",
        "depth": 3,
        "factory": lambda rng, N=N, d=3: make_haar(N, depth=d, rng=rng),
    },
    {
        "key": "haar_L5",
        "label": "Haar L=5",
        "depth": 5,
        "factory": lambda rng, N=N, d=5: make_haar(N, depth=d, rng=rng),
    },
    {
        "key": "brickwork_L3",
        "label": "Brickwork L=3",
        "depth": 3,
        "factory": lambda rng, N=N, d=3: make_brickwork(N, depth=d, rng=rng),
    },
    {
        "key": "brickwork_L5",
        "label": "Brickwork L=5",
        "depth": 5,
        "factory": lambda rng, N=N, d=5: make_brickwork(N, depth=d, rng=rng),
    },
    # {
    #     "key": "ucc-1",
    #     "label": "UCC-1",
    #     "depth": 1,
    #     "factory": lambda rng, N=N, d=1: make_uccsd(N, rng=rng, n_trotter_steps=d),
    # },
]

# Truncation methods compared
METHODS = ["top-k", "random-k", "schmidt-1cut", "schmidt-3cut"]

METHOD_STYLE = {
    "top-k": {"color": "#0C5DA5", "ls": "-", "mk": "o", "label": "Top-k (baseline)"},
    "random-k": {"color": "#888888", "ls": ":", "mk": "x", "label": "Random-k"},
    "schmidt-1cut": {
        "color": "#E87722",
        "ls": "--",
        "mk": "s",
        "label": "Schmidt-1cut",
    },
    "schmidt-3cut": {
        "color": "#D71111",
        "ls": "-.",
        "mk": "^",
        "label": "Schmidt-3cut",
    },
}

print(f"N={N}, k={K}, trials={N_TRIALS}")
print(f"Circuits: {[c['label'] for c in CIRCUIT_CONFIGS]}")

In [ ]:
# ── Precisely specified Schmidt truncation ────────────────────────────────────
#
# Bipartition convention (matches sparse_state.py bit ordering):
#   qubit q → bit position q in the integer x
#   qubit 0 = bit 0 (LSB),  qubit N-1 = bit N-1 (MSB)
#
# For bipartition at qubit `cut`:
#   LEFT partition:  qubits 0 .. cut-1   (lower bits, x & ((1<<cut)-1))
#   RIGHT partition: qubits cut .. N-1   (upper bits, x >> cut)
#
# Schmidt sector = RIGHT partition value = x >> cut
#   States sharing the same right-partition value form one Schmidt sector.
#   C(x_i) = #{j : (x_j >> cut) == (x_i >> cut)} among stored states.
#
# Schmidt-1cut score:  s_i = |alpha_i|² / C_i              [central cut = N//2]
# Schmidt-3cut score:  s_i = |alpha_i|² / (C1_i C2_i C3_i)^(1/3)
#   cuts at N//4, N//2, 3N//4

RADIX_THRESHOLD = 2000  # switch argsort → radix_sort


class TruncationSimulator:
    """
    Sparse-state simulator with pluggable truncation strategy.

    Shares the same Numba gate-application kernels as FixedBasisSimulator /
    SparseState, but allows injection of custom truncation methods after
    each gate application.  Normalization convention: after every truncation
    step, sum(|alpha_i|²) = 1 and gamma *= sqrt(retained_prob).

    Parameters
    ----------
    N          : system size
    k          : sparse budget (target number of retained basis states)
    truncation : one of 'top-k', 'random-k', 'schmidt-1cut', 'schmidt-3cut'
    """

    CUTS_1 = None  # set in __init__ to N//2
    CUTS_3 = None  # set in __init__ to [N//4, N//2, 3*N//4]

    def __init__(self, N, k, truncation="top-k"):
        self.N = N
        self.k = k
        self.truncation = truncation
        self.CUTS_1 = N // 2
        self.CUTS_3 = [N // 4, N // 2, 3 * N // 4]

        cap = 8 * k  # working buffer
        self.x = np.zeros(cap, dtype=np.uint64)
        self.alpha = np.zeros(cap, dtype=np.complex128)
        self._tx = np.zeros(4 * cap, dtype=np.uint64)
        self._ta = np.zeros(4 * cap, dtype=np.complex128)
        self.nnz = 0
        self.gamma = 1.0

    # ── State initialisation ──────────────────────────────────────────────────

    def reset(self):
        """Reset to |0...0> with gamma = 1."""
        self.x[:] = 0
        self.alpha[:] = 0.0
        self.x[0] = np.uint64(0)
        self.alpha[0] = 1.0 + 0j
        self.nnz = 1
        self.gamma = 1.0

    # ── Gate application ──────────────────────────────────────────────────────

    def apply_gate(self, gate):
        if gate.n_qubits == 1:
            self._apply_1q(gate)
        elif gate.n_qubits == 2:
            self._apply_2q(gate)
        else:
            raise NotImplementedError
        if self.nnz > self.k:
            self._truncate()

    def _sort_merge(self, tnnz):
        """In-place sort+merge of temp buffer (same logic as SparseState)."""
        if tnnz < RADIX_THRESHOLD:
            order = np.argsort(self._tx[:tnnz])
            self._tx[:tnnz] = self._tx[order]
            self._ta[:tnnz] = self._ta[order]
        else:
            radix_sort_uint64(self._tx, self._ta, tnnz)
        return merge_sorted_kernel(self._tx, self._ta, tnnz)

    def _copy_back(self, tnnz):
        if tnnz > self.x.shape[0]:
            new_cap = max(tnnz, 2 * self.x.shape[0])
            self.x = np.zeros(new_cap, dtype=np.uint64)
            self.alpha = np.zeros(new_cap, dtype=np.complex128)
        self.x[:tnnz] = self._tx[:tnnz]
        self.alpha[:tnnz] = self._ta[:tnnz]
        self.nnz = tnnz

    def _apply_1q(self, gate):
        q = gate.qubits[0]
        needed = 2 * self.nnz
        if needed > self._tx.shape[0]:
            self._tx = np.zeros(needed * 2, dtype=np.uint64)
            self._ta = np.zeros(needed * 2, dtype=np.complex128)
        tnnz = apply_1qubit_gate_kernel_opt(
            self.x, self.alpha, self.nnz, self._tx, self._ta, gate.matrix, q
        )
        tnnz = self._sort_merge(tnnz)
        self._copy_back(tnnz)

    def _apply_2q(self, gate):
        q1, q2 = gate.qubits
        needed = 4 * self.nnz
        if needed > self._tx.shape[0]:
            self._tx = np.zeros(needed * 2, dtype=np.uint64)
            self._ta = np.zeros(needed * 2, dtype=np.complex128)
        tnnz = apply_2qubit_gate_kernel_opt(
            self.x, self.alpha, self.nnz, self._tx, self._ta, gate.matrix, q1, q2
        )
        if tnnz == -1:
            raise RuntimeError("Temp buffer overflow in 2q gate")
        tnnz = self._sort_merge(tnnz)
        self._copy_back(tnnz)

    # ── Truncation ────────────────────────────────────────────────────────────

    def _select_and_normalise(self, idx):
        """
        Given selected indices idx (size k), copy to front, update gamma,
        and renormalise so sum(|alpha|²) == 1.
        Returns retained gamma_sq for recording.
        """
        probs_full = np.abs(self.alpha[: self.nnz]) ** 2
        gamma_sq = float(np.sum(probs_full[idx]))
        gamma_layer = np.sqrt(max(gamma_sq, 1e-300))

        temp_x = self.x[idx].copy()
        temp_a = self.alpha[idx].copy() / gamma_layer

        self.x[: self.k] = temp_x
        self.alpha[: self.k] = temp_a
        self.nnz = self.k
        self.gamma *= gamma_layer
        return gamma_sq

    def _truncate(self):
        t = self.truncation
        if t == "top-k":
            self._trunc_topk()
        elif t == "random-k":
            self._trunc_random()
        elif t == "schmidt-1cut":
            self._trunc_schmidt(cuts=[self.CUTS_1])
        elif t == "schmidt-3cut":
            self._trunc_schmidt(cuts=self.CUTS_3)
        else:
            raise ValueError(f"Unknown truncation: {t!r}")

    def _trunc_topk(self):
        probs = np.abs(self.alpha[: self.nnz]) ** 2
        kth = self.nnz - self.k
        idx = np.argpartition(probs, kth)[kth:]
        self._select_and_normalise(idx)

    def _trunc_random(self):
        idx = np.random.choice(self.nnz, size=self.k, replace=False)
        self._select_and_normalise(idx)

    def _trunc_schmidt(self, cuts):
        """
        Schmidt-weighted truncation.

        For each cut c in `cuts`:
        sector_c(x_i) = x_i >> c   (right-partition bits above qubit c)
        C_c(i) = |{j : x_j >> c == x_i >> c}|  (sector occupancy)

        Single cut:  score_i = |alpha_i|² / C_c(i)
        Multiple cuts: score_i = |alpha_i|² / geomean(C_c1, C_c2, ..., Cc_k)
                            = |alpha_i|² / (∏_c C_c(i))^(1/len(cuts))

        Rationale (matches paper's intent):
        High sector count → over-represented sector → penalised.
        Low sector count  → under-represented sector → boosted.
        Top-k by score therefore diversifies across Schmidt sectors while
        sacrificing some retained probability.
        """
        n = self.nnz
        xs = self.x[:n]
        probs = np.abs(self.alpha[:n]) ** 2

        log_combined_count = np.zeros(n, dtype=float)  # sum of log(C_c)
        for cut in cuts:
            sectors = (xs >> np.uint64(cut)).astype(np.int64)
            _, inverse = np.unique(sectors, return_inverse=True)
            counts = np.bincount(inverse)  # count per sector
            log_combined_count += np.log(np.maximum(counts[inverse], 1).astype(float))

        # Geometric mean of counts = exp(mean(log(counts)))
        n_cuts = len(cuts)
        scores = probs / np.exp(log_combined_count / n_cuts)

        kth = n - self.k
        idx = np.argpartition(scores, kth)[kth:]
        self._select_and_normalise(idx)

    # ── Full circuit simulation ───────────────────────────────────────────────

    def simulate(self, circuit, record_gamma=False):
        """
        Simulate the circuit and return the final SparseState-like object.

        Parameters
        ----------
        record_gamma : bool
            If True, record gamma after every gate application.

        Returns
        -------
        gamma_history : array of shape (len(circuit)+1,)  [only if record_gamma]
        """
        self.reset()
        if record_gamma:
            gh = [1.0]
        for gate in circuit:
            self.apply_gate(gate)
            if record_gamma:
                gh.append(self.gamma)
        if record_gamma:
            return np.array(gh)

    def fidelity(self, exact_sv):
        """Overlap |<exact|approx>|² with the current sparse state."""
        xs = self.x[: self.nnz]
        a = self.alpha[: self.nnz]
        overlap = np.dot(exact_sv[xs].conj(), a)
        return float(abs(overlap) ** 2)


print("TruncationSimulator defined.")
print()
# ── Quick validation ──────────────────────────────────────────────────────────
print("Validation: top-k must always achieve higher score than Schmidt-1cut")
print("  (by construction — Schmidt-1cut may select lower-probability states)")

In [ ]:
# ── Implementation validation ─────────────────────────────────────────────────
# Run all 4 methods on the same Haar L=2 circuit; verify:
# 1. top-k always retains highest gamma_sq at each step
# 2. Schmidt results are reproducible (deterministic given circuit)
# 3. Final gamma values are <= 1
# 4. Normalisation is preserved: sum(|alpha|^2) == 1 after truncation

rng_v = np.random.default_rng(9999)
val_circuit = make_haar(N, depth=2, rng=rng_v)
exact_sv_v = exact_statevector(N, val_circuit)

gammas_v = {}
fids_v = {}
for method in METHODS:
    sim = TruncationSimulator(N, K, truncation=method)
    gh = sim.simulate(val_circuit, record_gamma=True)
    gammas_v[method] = gh
    fids_v[method] = sim.fidelity(exact_sv_v)
    norm = float(np.sum(np.abs(sim.alpha[: sim.nnz]) ** 2))
    print(
        f"  {method:<14}: gamma={gh[-1]:.6f}  F={fids_v[method]:.4e}  "
        f"norm={norm:.8f}  nnz={sim.nnz}"
    )

print()
print("Assertions:")
print(
    f"  top-k gamma >= schmidt-1cut gamma: "
    f"{gammas_v['top-k'][-1] >= gammas_v['schmidt-1cut'][-1] - 1e-10}"
)
print(f"  All gamma <= 1.0: " f"{all(g[-1] <= 1.0 + 1e-10 for g in gammas_v.values())}")
print(
    f"  top-k fidelity >= schmidt-3cut: "
    f"{fids_v['top-k'] >= fids_v['schmidt-3cut'] - 1e-12}"
)

In [ ]:
# ── Main benchmark: 100 trials × 4 methods × 5 circuit families ──────────────
_npz = DATA_DIR / "schmidt_benchmark.npz"
_meta = DATA_DIR / "schmidt_benchmark_meta.json"
_exp = {
    "N": N,
    "K": K,
    "n_trials": N_TRIALS,
    "seed_base": SEED_BASE,
    "circuits": [c["key"] for c in CIRCUIT_CONFIGS],
    "methods": METHODS,
}


def run_benchmark():
    n_circ = len(CIRCUIT_CONFIGS)
    n_meth = len(METHODS)
    # fidelity[circuit_idx, method_idx, trial]
    fidelity = np.full((n_circ, n_meth, N_TRIALS), np.nan)
    gamma_sq = np.full((n_circ, n_meth, N_TRIALS), np.nan)

    for ci, cfg in enumerate(CIRCUIT_CONFIGS):
        print(f"\n[{cfg['label']}]  {N_TRIALS} trials × {len(METHODS)} methods")
        for trial in range(N_TRIALS):
            seed = SEED_BASE * 10**6 + ci * 10_000 + trial
            rng = np.random.default_rng(seed)
            # Same circuit for ALL methods (paired comparison)
            circuit = cfg["factory"](rng)
            exact_sv = exact_statevector(N, circuit)

            for mi, method in enumerate(METHODS):
                sim = TruncationSimulator(N, K, truncation=method)
                sim.simulate(circuit)
                fidelity[ci, mi, trial] = sim.fidelity(exact_sv)
                gamma_sq[ci, mi, trial] = sim.gamma**2

            if trial % 20 == 0:
                topk_f = fidelity[ci, 0, trial]
                s1_f = fidelity[ci, 2, trial]
                ratio = (s1_f / topk_f) if topk_f > 1e-300 else np.nan
                print(
                    f"  trial {trial:3d}: top-k F={topk_f:.3e}  "
                    f"s1cut F={s1_f:.3e}  ratio={ratio:.3f}"
                )

    return {"fidelity": fidelity, "gamma_sq": gamma_sq}


if _npz.exists() and _meta.exists() and not FORCE_RERUN:
    stored = json.loads(_meta.read_text())
    if stored == _exp:
        raw = dict(np.load(_npz, allow_pickle=False))
        print(f"Loaded cache: {_npz}")
    else:
        print("Cache mismatch — recomputing")
        raw = run_benchmark()
        np.savez_compressed(_npz, **raw)
        _meta.write_text(json.dumps(_exp, indent=2))
else:
    raw = run_benchmark()
    np.savez_compressed(_npz, **raw)
    _meta.write_text(json.dumps(_exp, indent=2))

fidelity_raw = raw["fidelity"]  # (n_circ, n_meth, N_TRIALS)
gamma_sq_raw = raw["gamma_sq"]  # same shape
print(f"\nData shape: {fidelity_raw.shape}")

In [ ]:
# ── Rigorous statistics table ─────────────────────────────────────────────────
# For every (circuit, comparison method) pair:
#   - Paired per-trial ratio: fidelity_method[trial] / fidelity_topk[trial]
#   - Geometric mean + 95% bootstrap CI of ratios
#   - Median fidelity for each method [IQR]
#   - Win rate (top-k > method) with Wilson binomial CI
#   - Wilcoxon signed-rank p-value (paired, one-sided: top-k > method)

from scipy.stats import wilcoxon


def wilson_ci(k, n, ci=0.95):
    """Wilson score CI for binomial proportion."""
    if n == 0:
        return np.nan, np.nan
    z = sci_stats.norm.ppf(0.5 + ci / 2)
    p = k / n
    denom = 1 + z**2 / n
    ctr = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return float(max(0, ctr - margin)), float(min(1, ctr + margin))


topk_idx = METHODS.index("top-k")

print("\n" + "=" * 105)
print(f"Schmidt-Weighted Truncation Benchmark  N={N}, k={K}, {N_TRIALS} trials")
print("=" * 105)
hdr = (
    f"{'Circuit':<16} {'Method':<15}  "
    f"{'F_topk  med[IQR]':<26}  "
    f"{'F_meth  med[IQR]':<26}  "
    f"{'Ratio gm[CI95]':<20}  "
    f"{'TopK>Meth':>10}  "
    f"{'p-val':>7}"
)
print(hdr)
print("-" * 105)

all_stats = {}

for ci, cfg in enumerate(CIRCUIT_CONFIGS):
    topk_f = fidelity_raw[ci, topk_idx, :]

    for mi, method in enumerate(METHODS):
        if method == "top-k":
            continue
        meth_f = fidelity_raw[ci, mi, :]

        # Per-trial ratios (only where top-k > 0)
        valid = topk_f > 1e-300
        ratios = meth_f[valid] / topk_f[valid]
        rs = trial_stats(ratios, n_boot=4000)

        # Fidelity stats
        fs_topk = trial_stats(topk_f, n_boot=4000)
        fs_meth = trial_stats(meth_f, n_boot=4000)

        # Win rate: how often top-k strictly beats method
        wins = int(np.sum(topk_f > meth_f))
        wci_lo, wci_hi = wilson_ci(wins, N_TRIALS)

        # Wilcoxon signed-rank (paired, one-sided: top-k > method)
        if np.any(topk_f != meth_f):
            try:
                _, pval = wilcoxon(
                    topk_f, meth_f, alternative="greater", zero_method="wilcox"
                )
            except:
                pval = np.nan
        else:
            pval = np.nan

        all_stats[(cfg["key"], method)] = {
            "ratio_gm": rs["geomean"],
            "ratio_ci_lo": rs["gm_ci_lo"],
            "ratio_ci_hi": rs["gm_ci_hi"],
            "ratio_median": rs["median"],
            "f_topk_median": fs_topk["median"],
            "f_meth_median": fs_meth["median"],
            "win_rate": wins / N_TRIALS,
            "pval": pval,
        }

        def fmt_med_iqr(s, prec=".2e"):
            return f"{s['median']:{prec}} [{s['q25']:{prec}},{s['q75']:{prec}}]"

        def fmt_gm_ci(r):
            return f"{r['geomean']:.3f}x [{r['gm_ci_lo']:.3f},{r['gm_ci_hi']:.3f}]"

        print(
            f"{cfg['label']:<16} {method:<15}  "
            f"{fmt_med_iqr(fs_topk):<26}  "
            f"{fmt_med_iqr(fs_meth):<26}  "
            f"{fmt_gm_ci(rs):<20}  "
            f"{wins:>4}/{N_TRIALS} [{100*wci_lo:.0f}%,{100*wci_hi:.0f}%]  "
            f"p={pval:.3e}"
            if not np.isnan(pval)
            else f"{cfg['label']:<16} {method:<15}  "
            f"{fmt_med_iqr(fs_topk):<26}  "
            f"{fmt_med_iqr(fs_meth):<26}  "
            f"{fmt_gm_ci(rs):<20}  "
            f"{wins:>4}/{N_TRIALS} [{100*wci_lo:.0f}%,{100*wci_hi:.0f}%]  "
            f"p=  N/A"
        )

print("=" * 105)
print(
    "Ratio < 1 means method WORSE than top-k.  Win rate = fraction of trials where top-k > method."
)
print(
    "p-val: Wilcoxon signed-rank, one-sided (H1: top-k > method), Holm-corrected not shown."
)